In [3]:
import requests
import pandas as pd
from datetime import datetime

# ==========================================
# CONFIGURATION & SETUP
# ==========================================
# TODO: Replace with your actual OpenWeather API key
API_KEY = "62f560708d6f7d9df103aa256c4fdea9"
CITIES = ["Lagos", "Portharcourt", "Abuja", "Maiduguri", "Kano", "Enugu"]
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

# ==========================================
# TASK 1: EXTRACT DATA
# ==========================================
def extract_weather_data(cities, api_key):
    raw_data = []
    print("--- Starting Extraction ---")
    
    for city in cities:
        
        params = {
            'q': city,
            'appid': api_key,
            'units': 'metric' 
        }
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status() 
            
            data = response.json()
            raw_data.append(data)
            print(f"Successfully extracted data for {city}")
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for {city}: {e}")
            
    return raw_data

# ==========================================
# TASK 2: TRANSFORM DATA
# ==========================================
def transform_weather_data(raw_data_list):
    print("\n--- Starting Transformation ---")
    transformed_records = []
    
    for data in raw_data_list:
  
        record = {
            "City": data.get("name"),
            "Country": data.get("sys", {}).get("country"),
            "Temperature_C": data.get("main", {}).get("temp"),
            "Humidity_Percent": data.get("main", {}).get("humidity"),
            "Weather_Condition": data.get("weather", [{}])[0].get("main"),
            "Wind_Speed_m_s": data.get("wind", {}).get("speed"),
            # Converting UNIX timestamp to readable local datetime
            "Fetched_At": datetime.fromtimestamp(data.get("dt")).strftime('%Y-%m-%d %H:%M:%S')
        }
        transformed_records.append(record)
        
    # Converting list of dictionaries into a Pandas DataFrame
    df = pd.DataFrame(transformed_records)
    
    # Simple transformations: Ensuring correct data types
    df["Temperature_C"] = df["Temperature_C"].astype(float)
    df["Humidity_Percent"] = df["Humidity_Percent"].astype(int)
    df["Wind_Speed_m_s"] = df["Wind_Speed_m_s"].astype(float)
    
    print("Transformation completed successfully!")
    return df

# ==========================================
# TASK 3: LOAD DATA
# ==========================================
def load_data_to_csv(df, filename="weather_report.csv"):
    print("\n--- Starting Loading ---")
    try:
        # Save DataFrame to CSV without index
        df.to_csv(filename, index=False)
        print(f"Data successfully saved to {filename}!")
    except Exception as e:
        print(f"Error saving data: {e}")

# ==========================================
# MAIN EXECUTION RUNNER
# ==========================================
if __name__ == "__main__":
    # Step 1: Extract
    raw_weather = extract_weather_data(CITIES, API_KEY)
    
    if raw_weather:
        # Step 2: Transform
        cleaned_df = transform_weather_data(raw_weather)
        
        # Display the cleaned data in console
        print("\nCleaned Dataset Preview:")
        print(cleaned_df)
        
        # Step 3: Load
        load_data_to_csv(cleaned_df, "cleaned_weather_data.csv")
    else:
        print("No data extracted. Pipeline aborted.")

--- Starting Extraction ---
Successfully extracted data for Lagos
Error fetching data for Portharcourt: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Portharcourt&appid=62f560708d6f7d9df103aa256c4fdea9&units=metric
Successfully extracted data for Abuja
Successfully extracted data for Maiduguri
Successfully extracted data for Kano
Successfully extracted data for Enugu

--- Starting Transformation ---
Transformation completed successfully!

Cleaned Dataset Preview:
        City Country  Temperature_C  Humidity_Percent Weather_Condition  \
0      Lagos      NG          24.80                91              Rain   
1      Abuja      NG          23.68                91            Clouds   
2  Maiduguri      NG          31.43                47            Clouds   
3       Kano      NG          29.34                58            Clouds   
4      Enugu      NG          24.63                87            Clouds   

   Wind_Speed_m_s           Fetched_At  